# iFinD 新信号抓取 —— ETF 资金流 / 份额 / 期权隐含波动率

为策略的"方向2(接入新信号)"准备数据。目标抓两类**与现有量价正交**的信号:

1. **ETF 资金流 / 份额(横截面因子候选)**：ETF 总份额的变化 ≈ 净申购赎回 = 资金流入/流出。
   `资金流额 ≈ Δ份额 × 单位净值`。资金持续流入的 ETF 往往后续更强 —— 这是真正独立于价格动量的信息。
2. **期权隐含波动率(市场择时信号)**：50ETF/300ETF 期权的 IV 是"恐慌指标",可做风险开关。
   注意:只有少数 ETF 有期权,所以这是**市场级**信号,不是横截面因子。

> **关于指标代码的现实问题**：iFinD HTTP 手册没有完整列出"份额/资金流/IV"的英文指标代码
> (它让你去 **PC 端"超级命令"** 里生成)。所以本 notebook 用一个 **probe(探测)单元**:
> 拿一批候选指标名去实测,**哪个返回有效数据就用哪个**,你不必预先知道代码。
> 最权威的做法仍是:打开同花顺 iFinD「超级命令」,输入你要的数据(如"基金份额"),
> 复制它生成的指标串,填进 `CONFIRMED_*` 配置。

**复用了行情 notebook 的稳健框架**:token 自动续、600/min 限流、错误退避、断点续跑。


## 1. 配置

In [ ]:
# ====== 用户配置 ======
REFRESH_TOKEN = "在这里粘贴你的 refresh_token"   # ← 必填(可用之前那个,有效期至 2026-07-15)

START_DATE = "2016-01-01"
END_DATE   = "2026-06-12"

UNIVERSE_JSON = "/Users/shenboheng/Documents/ClaudeCode/dataset/基金深度分析/bulk_universe.json"
OUT_DIR = "/Users/shenboheng/Documents/ClaudeCode/dataset/基金深度分析"
FLOW_DB = OUT_DIR + "/etf_flow_ifind.db"        # 资金流/份额落地
IV_DB   = OUT_DIR + "/etf_iv_ifind.db"          # 期权 IV 落地

# 有期权的主要 ETF(市场级 IV 信号用;按需增减)
OPTION_UNDERLYINGS = ["510050.SH", "510300.SH", "159919.SZ", "510500.SH",
                      "159922.SZ", "588000.SH", "159915.SZ", "588080.SH"]

CODES_PER_REQUEST = 12
REQUESTS_PER_MIN  = 500
HTTP_TIMEOUT      = 180
MIN_INTERVAL = 60.0 / REQUESTS_PER_MIN


## 2. Token + 三个取数端点(历史行情 / 日期序列 / 基础数据)

In [ ]:
import requests, json, time
import pandas as pd

BASE = "https://quantapi.51ifind.com/api/v1"
_state = {"token": None, "last_req": 0.0}
QUOTA_CODES = {-4301, -4302, -4303, -4317, -4318}

class QuotaStop(Exception):
    pass

def refresh_access_token():
    r = requests.post(f"{BASE}/get_access_token",
                      headers={"Content-Type": "application/json", "refresh_token": REFRESH_TOKEN}, timeout=60)
    tok = (r.json().get("data") or {}).get("access_token")
    if not tok:
        raise RuntimeError(f"获取 access_token 失败: {r.json()}")
    _state["token"] = tok
    print("access_token OK:", tok[:12], "...")
    return tok

def _throttle():
    dt = time.time() - _state["last_req"]
    if dt < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - dt)
    _state["last_req"] = time.time()

def _post(endpoint, payload, _depth=0):
    if _state["token"] is None:
        refresh_access_token()
    headers = {"Content-Type": "application/json", "access_token": _state["token"]}
    _throttle()
    try:
        j = requests.post(f"{BASE}/{endpoint}", json=payload, headers=headers, timeout=HTTP_TIMEOUT).json()
    except Exception:
        if _depth >= 5:
            raise
        time.sleep(5 * (_depth + 1)); return _post(endpoint, payload, _depth + 1)
    ec = j.get("errorcode", j.get("errcode", 0))
    if ec in (0, None):
        return j
    if ec == -4001:
        return {"tables": [], "errorcode": -4001}
    if ec in (-1300, -1302) and _depth < 3:
        refresh_access_token(); return _post(endpoint, payload, _depth + 1)
    if ec in (-4102, -4101, -5002, -5004) and _depth < 6:
        time.sleep(6 * (_depth + 1)); return _post(endpoint, payload, _depth + 1)
    if ec == -4400 and _depth < 6:
        time.sleep(20); return _post(endpoint, payload, _depth + 1)
    if ec in QUOTA_CODES:
        raise QuotaStop(f"{ec}: {j.get('errmsg')}")
    if _depth < 4:
        time.sleep(5 * (_depth + 1)); return _post(endpoint, payload, _depth + 1)
    return j   # 把错误原样返回(probe 要看 errorcode)

def hist_quotation(codes, indicators, cps="1", start=START_DATE, end=END_DATE):
    return _post("cmd_history_quotation", {
        "codes": ",".join(codes), "indicators": ",".join(indicators) if isinstance(indicators, list) else indicators,
        "startdate": start, "enddate": end, "functionpara": {"CPS": cps, "Fill": "Blank"}})

def date_sequence(codes, indicator, params=None, start=START_DATE, end=END_DATE):
    return _post("date_sequence", {
        "codes": ",".join(codes), "startdate": start, "enddate": end,
        "functionpara": {"Interval": "D", "Fill": "Blank"},
        "indipara": [{"indicator": indicator, "indicatorparams": params or [""]}]})

def basic_data(codes, indicator, params=None):
    return _post("basic_data_service", {
        "codes": ",".join(codes),
        "indipara": [{"indicator": indicator, "indicatorparams": params or [""]}]})

def parse_tables(j, suffix=""):
    frames = []
    for t in (j.get("tables") or []):
        code = t.get("thscode"); times = t.get("time") or []
        table = t.get("table") or {}
        if not times:
            continue
        d = {"thscode": code, "date": times}
        for k, v in table.items():
            v = list(v)
            if len(v) < len(times):
                v = v + [None] * (len(times) - len(v))
            d[k + suffix] = v[:len(times)]
        frames.append(pd.DataFrame(d))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if not REFRESH_TOKEN.startswith("在这里"):
    refresh_access_token()


## 3. 探测(probe):哪些指标名能取到数据？

下面对一批**候选指标名**逐个实测(取最近一小段日期、单只代码)。看 `errorcode==0` 且有非空值的,
就是可用的。把可用的填进第 4/5 节的 `CONFIRMED_*`。

> 这些候选名是**经验猜测**,不保证;真不行就去「超级命令」复制准确代码。
> 三个端点都试:`date_sequence`(日序列,份额类多在这)、`hist_quotation`(行情)、`basic_data`(基础)。

In [ ]:
def _has_data(j):
    if j.get("errorcode", j.get("errcode", 0)) not in (0, None):
        return False, f"ec={j.get('errorcode', j.get('errcode'))}"
    df = parse_tables(j)
    valcols = [c for c in df.columns if c not in ("thscode", "date")]
    if df.empty or not valcols:
        return False, "无表"
    nonnull = int(pd.to_numeric(df[valcols[0]], errors="coerce").notna().sum())
    return (nonnull > 0), f"非空{nonnull}/{len(df)}"

def probe(candidates, test_code, kind):
    """candidates: list[str] 指标名; 对 date_sequence / hist / basic 各试一遍。"""
    print(f"\n=== probe [{kind}] code={test_code} (近30天) ===")
    s = (pd.Timestamp(END_DATE) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")
    for ind in candidates:
        for ep, fn in [("date_seq", lambda: date_sequence([test_code], ind, start=s)),
                       ("hist",     lambda: hist_quotation([test_code], ind, start=s)),
                       ("basic",    lambda: basic_data([test_code], ind))]:
            try:
                ok, msg = _has_data(fn())
            except QuotaStop:
                print("  额度耗尽,停"); return
            except Exception as e:
                ok, msg = False, f"异常{type(e).__name__}"
            flag = "✅可用" if ok else "  ✗"
            print(f"  {flag}  {ep:9} {ind:38} {msg}")

# 份额 / 资金流 候选(经验猜测,以 probe 结果为准)
# 基金类指标命名约定 = ths_<概念>_fund(已从 iFinD 指标库确认:ths_unit_nav_fund/ths_acc_nav_fund/
# ths_total_asset_fund 等)。下列按约定列候选,probe 实测哪个可用。
# 份额取不到时,ths_total_asset_fund(基金规模/元,已确认可用)是兜底:规模÷净值≈份额。
SHARE_CANDIDATES = [
    "ths_total_share_fund", "ths_share_total_fund", "ths_unit_total_fund",
    "ths_fund_share", "ths_share_fund", "ths_circulating_share_fund",
    "ths_total_asset_fund",                       # 基金规模(元)·已确认可用·兜底
    "ths_net_purchase_redeem_fund", "ths_net_subscribe_redeem_fund",
]
# 期权 IV 候选(在期权标的或波指上试)
# IV 在 ETF 标的上多半取不到(它是期权合约属性);ths_volatility_fund 是"已实现"年化波动(非隐含),可做风险兜底。
IV_CANDIDATES = [
    "ths_implied_vol_option", "ths_iv_option", "ths_implied_volatility_option",
    "ths_atm_implied_vol_option", "ths_vix_index", "ths_volatility_fund",
]

probe(SHARE_CANDIDATES, "510300.SH", "ETF份额/资金流")
probe(IV_CANDIDATES, "510050.SH", "期权IV")


## 4. 抓 ETF 资金流 / 份额(全市场,横截面因子用)

把上面 probe 里**可用**的份额指标填进 `CONFIRMED_SHARE`,并设对应端点。然后跑全量。
落地后:`资金流 ≈ Δ份额 × 单位净值`(单位净值用行情 notebook 已抓的 `netAssetValue`)。

In [ ]:
# ← 用 probe 结果填写：指标名 + 端点("date_seq"/"hist"/"basic")
CONFIRMED_SHARE = {"indicator": "ths_total_share_fund", "endpoint": "date_seq"}

from pathlib import Path
import sqlite3

raw_uni = json.loads(Path(UNIVERSE_JSON).read_text(encoding="utf-8"))["data"]
def to_thscode(c):
    return c + ".SH" if c.startswith("5") else (c + ".SZ" if c.startswith("1") else None)
codes = sorted({to_thscode(c) for c, it in raw_uni.items()
                if (("ETF" in str(it.get("name", "")).upper()) or ("交易型开放式" in str(it.get("name", ""))))
                and "联接" not in str(it.get("name", "")) and "FOF" not in str(it.get("name", "")).upper()
                and to_thscode(c)})
print(f"目标 ETF: {len(codes)} 只")

def fetch_share(batch):
    ind, ep = CONFIRMED_SHARE["indicator"], CONFIRMED_SHARE["endpoint"]
    j = (date_sequence(batch, ind) if ep == "date_seq" else
         hist_quotation(batch, ind) if ep == "hist" else basic_data(batch, ind))
    df = parse_tables(j)
    if not df.empty:
        df = df.rename(columns={c: "share" for c in df.columns if c not in ("thscode", "date")})
    return df

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def run_share(resume=True):
    done = set()
    if resume:
        try:
            con = sqlite3.connect(FLOW_DB)
            done = set(pd.read_sql_query("SELECT DISTINCT thscode FROM etf_share", con)["thscode"]); con.close()
        except Exception:
            pass
    todo = [c for c in codes if c not in done]
    print(f"待抓 {len(todo)} / {len(codes)}（已完成 {len(done)}）")
    con = sqlite3.connect(FLOW_DB)
    try:
        for bi, b in enumerate(chunks(todo, CODES_PER_REQUEST), 1):
            try:
                df = fetch_share(b)
            except QuotaStop as e:
                print(f"⏸ 额度耗尽，停：{e}（重跑续传）"); break
            if not df.empty:
                df["share"] = pd.to_numeric(df["share"], errors="coerce")
                df["fund_code"] = df["thscode"].str.split(".").str[0]
                df.to_sql("etf_share", con, if_exists="append", index=False)
            print(f"[{bi}] {b[0]}..{b[-1]} 写 {0 if df is None else len(df)} 行")
    finally:
        con.close()
    print(f"✅ -> {FLOW_DB}")

# run_share()   # ← 确认 CONFIRMED_SHARE 可用后取消注释运行
print("确认 CONFIRMED_SHARE 后取消最后一行注释运行")


## 5. 抓期权隐含波动率(市场择时信号)

把 probe 里可用的 IV 指标填进 `CONFIRMED_IV`。若**标的上直接取不到 IV**(IV 是期权合约属性,
不在 ETF 标的上),则需要先取期权合约链再逐合约取 IV——较复杂,本节先按"标的/波指能直接取"处理,
取不到就只能走合约链(见末尾说明)。

In [ ]:
CONFIRMED_IV = {"indicator": "ths_implied_vol_option", "endpoint": "date_seq"}

def fetch_iv(batch):
    ind, ep = CONFIRMED_IV["indicator"], CONFIRMED_IV["endpoint"]
    j = (date_sequence(batch, ind) if ep == "date_seq" else
         hist_quotation(batch, ind) if ep == "hist" else basic_data(batch, ind))
    df = parse_tables(j)
    if not df.empty:
        df = df.rename(columns={c: "iv" for c in df.columns if c not in ("thscode", "date")})
    return df

def run_iv():
    con = sqlite3.connect(IV_DB)
    try:
        for b in chunks(OPTION_UNDERLYINGS, CODES_PER_REQUEST):
            try:
                df = fetch_iv(b)
            except QuotaStop as e:
                print(f"⏸ 额度耗尽：{e}"); break
            if not df.empty:
                df["iv"] = pd.to_numeric(df["iv"], errors="coerce")
                df.to_sql("etf_iv", con, if_exists="replace" if b is OPTION_UNDERLYINGS[:CODES_PER_REQUEST] else "append", index=False)
            print(f"{b}: {0 if df is None else len(df)} 行")
    finally:
        con.close()
    print(f"✅ -> {IV_DB}")

# run_iv()   # ← 确认 CONFIRMED_IV 可用后取消注释运行
print("确认 CONFIRMED_IV 后取消最后一行注释运行")


## 6. 落地后怎么做成因子(下一步,在策略里)

**资金流因子(横截面,可直接进打分):**
```
份额面板 share[date, code]  →  资金流额 flow = share.diff() * netAssetValue
                            →  flow_20 = flow 近20日累计 / 近20日成交额(标准化)
                            →  z(flow_20) 作为新因子,先做 §3 因子审查(Rank IC/相关/衰减),
                               再 walk-forward 验证能否真正提升,过了才进合成分。
```
**期权 IV 信号(市场择时):**
```
IV(50ETF/300ETF) 高于其滚动分位 → 降低风险暴露(类似弱市择时,但用 IV 而非 MA200)。
注意:之前 MA200 弱市择时被证伪(whipsaw),IV 择时也必须 walk-forward 验证,别想当然。
```

**纪律(重要)**:任何新因子都要走 `factor_diagnostics.py`(IC/分层/相关/衰减)+ `walkforward_hfq.py`
(真样本外)。本项目反复证明:**样本内好看、OOS 站不住的改动一律不上。**

---
**附:若期权 IV 在标的上取不到(常见)** —— IV 属于期权合约,需:
1. 用 `basic_data`/数据池取 50ETF 期权的**合约代码**(当月平值附近);
2. 逐合约用 `hist_quotation`/`date_sequence` 取 IV;
3. 按到期日/在值程度加权成"市场 ATM IV"序列。
这步较重,确认要做的话我再单独给你写。
